
# Fill Shelter Capacity and Save SQLite DB

This notebook uses only 4 CSV files.

- `final_shelter_dataset.csv`: base shelter table
- `earthquake_shelter_clean_2.csv`: reference file
- `tsunami_shelter_clean_2.csv`: reference file
- `danger_clean.csv`: alert data for the DB

Goals:

1. Fill only missing values in the capacity column
2. Keep existing capacity values as they are
3. Leave values empty if both reference files do not have data
4. Save the final CSV and a simple SQLite DB


In [5]:

import os

import pandas as pd
from sqlalchemy import Float, ForeignKey, Integer, String, create_engine
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column


# Read CSV safely with a few common encodings.
def read_csv_safely(path: str) -> pd.DataFrame:
    for encoding in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return pd.read_csv(path, encoding=encoding)
        except UnicodeDecodeError:
            continue

    raise UnicodeDecodeError("unknown", b"", 0, 1, f"Could not read file: {path}")


# Find the project root with os.path only, then join the relative path.
def get_project_path(relative_path: str) -> str:
    current_dir = os.getcwd()

    while True:
        marker_dir = os.path.join(current_dir, "preprocessing_data", "preprocessing")
        if os.path.isdir(marker_dir):
            return os.path.join(current_dir, relative_path)

        parent_dir = os.path.dirname(current_dir)
        if parent_dir == current_dir:
            raise FileNotFoundError(f"Could not find project path for: {relative_path}")

        current_dir = parent_dir


final_df = read_csv_safely(get_project_path("preprocessing_data/preprocessing/final_shelter_dataset.csv"))
earthquake_df = read_csv_safely(get_project_path("preprocessing_data/preprocessing/earthquake_shelter_clean_2.csv"))
tsunami_df = read_csv_safely(get_project_path("preprocessing_data/preprocessing/tsunami_shelter_clean_2.csv"))
danger_df = read_csv_safely(get_project_path("preprocessing_data/preprocessing/danger_clean.csv"))


In [6]:

# Clean column names and string values.
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(col).strip() for col in df.columns]

    for column in df.columns:
        if df[column].dtype == "object":
            df[column] = df[column].map(
                lambda value: " ".join(str(value).split()) if pd.notna(value) else pd.NA
            )
            df[column] = df[column].replace("", pd.NA)

    return df


# Convert a column to numeric values.
def to_numeric_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df = df.copy()
    df[column] = pd.to_numeric(df[column], errors="coerce")
    return df


final_df = clean_dataframe(final_df)
earthquake_df = clean_dataframe(earthquake_df)
tsunami_df = clean_dataframe(tsunami_df)
danger_df = clean_dataframe(danger_df)

final_df = to_numeric_column(final_df, "수용인원")
earthquake_df = to_numeric_column(earthquake_df, "수용인원")
tsunami_df = to_numeric_column(tsunami_df, "수용인원")

required_columns = {
    "final_df": ["대피소명", "주소", "수용인원"],
    "earthquake_df": ["대피소명", "주소", "수용인원"],
    "tsunami_df": ["대피소명", "주소", "수용인원"],
    "danger_df": ["발표시간", "지역", "시군구", "재난종류", "특보등급", "해당지역"],
}

for label, columns in required_columns.items():
    current_df = locals()[label]
    missing_columns = [column for column in columns if column not in current_df.columns]
    if missing_columns:
        raise ValueError(f"Missing columns in {label}: {missing_columns}")



## Match Rule

This notebook matches shelters by using two columns together.

- shelter name column
- address column

Using both columns is simpler and safer than using only one of them.


In [7]:

# Fill missing capacities using two real columns directly.
match_columns = ["대피소명", "주소"]

earthquake_capacity = (
    earthquake_df[match_columns + ["수용인원"]]
    .dropna(subset=["수용인원"])
    .drop_duplicates(subset=match_columns)
    .rename(columns={"수용인원": "earthquake_capacity"})
)

tsunami_capacity = (
    tsunami_df[match_columns + ["수용인원"]]
    .dropna(subset=["수용인원"])
    .drop_duplicates(subset=match_columns)
    .rename(columns={"수용인원": "tsunami_capacity"})
)

conflicts = earthquake_capacity.merge(tsunami_capacity, on=match_columns, how="inner")
conflicts = conflicts[conflicts["earthquake_capacity"] != conflicts["tsunami_capacity"]]

final_filled_df = final_df.copy()

fill_result = final_filled_df.merge(earthquake_capacity, on=match_columns, how="left")
fill_result = fill_result.merge(tsunami_capacity, on=match_columns, how="left")

fill_mask = fill_result["수용인원"].isna()
fill_result.loc[fill_mask, "수용인원"] = fill_result.loc[fill_mask, "earthquake_capacity"]

fill_mask = fill_result["수용인원"].isna()
fill_result.loc[fill_mask, "수용인원"] = fill_result.loc[fill_mask, "tsunami_capacity"]

fill_result["수용인원"] = pd.to_numeric(
    fill_result["수용인원"],
    errors="coerce",
).astype("Int64")

final_filled_df = fill_result.drop(columns=["earthquake_capacity", "tsunami_capacity"])

saved_csv_file = get_project_path("preprocessing_data/preprocessing/final_shelter_dataset_filled_capacity.csv")
try:
    final_filled_df.to_csv(saved_csv_file, index=False, encoding="utf-8-sig")
except PermissionError:
    saved_csv_file = get_project_path("preprocessing_data/preprocessing/final_shelter_dataset_filled_capacity_copy.csv")
    final_filled_df.to_csv(saved_csv_file, index=False, encoding="utf-8-sig")



## Simple DB Structure

This notebook builds 5 tables.

1. `regions`
2. `disaster_types`
3. `danger_alerts`
4. `shelters`
5. `shelter_disaster_relations`

The goal is to keep the structure simple and easy to read.


In [8]:

# Save the SQLite DB using the requested 5-table structure.
# Use DataFrame merges directly instead of mapping dictionaries.


class Base(DeclarativeBase):
    pass


class Region(Base):
    __tablename__ = "regions"

    region_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    sido: Mapped[str] = mapped_column("시도", String, nullable=False)
    sigungu: Mapped[str] = mapped_column("시군구", String, nullable=False)


class DisasterType(Base):
    __tablename__ = "disaster_types"

    disaster_type_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    disaster_name: Mapped[str] = mapped_column("재난유형", String, nullable=False)


class DangerAlert(Base):
    __tablename__ = "danger_alerts"

    alert_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    announced_at: Mapped[str] = mapped_column("발표시간", String, nullable=False)
    region_id: Mapped[int] = mapped_column(ForeignKey("regions.region_id"), nullable=False)
    disaster_type_id: Mapped[int] = mapped_column(ForeignKey("disaster_types.disaster_type_id"), nullable=False)
    alert_level: Mapped[str | None] = mapped_column("특보등급", String, nullable=True)
    affected_area: Mapped[str | None] = mapped_column("해당지역", String, nullable=True)


class Shelter(Base):
    __tablename__ = "shelters"

    shelter_id: Mapped[int] = mapped_column(Integer, primary_key=True)
    region_id: Mapped[int] = mapped_column(ForeignKey("regions.region_id"), nullable=False)
    shelter_name: Mapped[str] = mapped_column("대피소명", String, nullable=False)
    address: Mapped[str] = mapped_column("주소", String, nullable=False)
    shelter_type: Mapped[str | None] = mapped_column("대피소유형", String, nullable=True)
    latitude: Mapped[float | None] = mapped_column("위도", Float, nullable=True)
    longitude: Mapped[float | None] = mapped_column("경도", Float, nullable=True)
    earthquake_note: Mapped[str | None] = mapped_column("지진설명", String, nullable=True)
    capacity: Mapped[int | None] = mapped_column("수용인원", Integer, nullable=True)


class ShelterDisasterRelation(Base):
    __tablename__ = "shelter_disaster_relations"

    shelter_id: Mapped[int] = mapped_column(ForeignKey("shelters.shelter_id"), primary_key=True)
    disaster_type_id: Mapped[int] = mapped_column(ForeignKey("disaster_types.disaster_type_id"), primary_key=True)


# Simple helpers for missing values.
def none_if_na(value):
    return None if pd.isna(value) else value


def int_if_na(value):
    return None if pd.isna(value) else int(value)


def float_if_na(value):
    return None if pd.isna(value) else float(value)


def find_disaster_types_from_shelter_type(value) -> list[str]:
    disaster_names = []
    shelter_type = "" if pd.isna(value) else str(value)

    if "한파쉼터" in shelter_type:
        disaster_names.append("한파")
    if "무더위쉼터" in shelter_type:
        disaster_names.append("폭염")
    if "지진옥외대피장소" in shelter_type:
        disaster_names.append("지진")
    if "지진해일대피장소" in shelter_type:
        disaster_names.append("지진해일")

    return disaster_names


db_shelters = final_filled_df.copy()
db_danger = danger_df.copy()

regions_from_shelters = db_shelters[["시도", "시군구"]].dropna().drop_duplicates()
regions_from_danger = (
    db_danger[["지역", "시군구"]]
    .rename(columns={"지역": "시도"})
    .dropna()
    .drop_duplicates()
)
regions = pd.concat([regions_from_shelters, regions_from_danger], ignore_index=True)
regions = regions.drop_duplicates().reset_index(drop=True)
regions.insert(0, "region_id", range(1, len(regions) + 1))

disaster_name_set = set(db_danger["재난종류"].dropna().astype(str).tolist())

if db_shelters["대피소유형"].fillna("").str.contains("한파쉼터").any():
    disaster_name_set.add("한파")
if db_shelters["대피소유형"].fillna("").str.contains("무더위쉼터").any():
    disaster_name_set.add("폭염")
if db_shelters["대피소유형"].fillna("").str.contains("지진옥외대피장소").any():
    disaster_name_set.add("지진")
if db_shelters["대피소유형"].fillna("").str.contains("지진해일대피장소").any():
    disaster_name_set.add("지진해일")

disaster_types = pd.DataFrame({"재난유형": sorted(disaster_name_set)})
disaster_types.insert(0, "disaster_type_id", range(1, len(disaster_types) + 1))

shelters_table = db_shelters.merge(regions, on=["시도", "시군구"], how="left")
shelters_table.insert(0, "shelter_id", range(1, len(shelters_table) + 1))
shelters_table = shelters_table.rename(columns={"지역": "지진설명"})
shelters_table = shelters_table[
    [
        "shelter_id",
        "region_id",
        "대피소명",
        "주소",
        "대피소유형",
        "위도",
        "경도",
        "지진설명",
        "수용인원",
    ]
].copy()

danger_regions = db_danger.rename(columns={"지역": "시도"})
danger_alerts = danger_regions.merge(regions, on=["시도", "시군구"], how="left")
danger_alerts = danger_alerts.merge(
    disaster_types,
    left_on="재난종류",
    right_on="재난유형",
    how="left",
)
danger_alerts.insert(0, "alert_id", range(1, len(danger_alerts) + 1))
danger_alerts = danger_alerts[
    ["alert_id", "발표시간", "region_id", "disaster_type_id", "특보등급", "해당지역"]
].copy()

type_relations = shelters_table[["shelter_id", "대피소유형"]].copy()
type_relations["재난유형"] = type_relations["대피소유형"].apply(find_disaster_types_from_shelter_type)
type_relations = type_relations.explode("재난유형")
type_relations = type_relations.dropna(subset=["재난유형"])
type_relations = type_relations[["shelter_id", "재난유형"]]

earthquake_relations = shelters_table[["shelter_id", "대피소명", "주소"]].merge(
    earthquake_df[["대피소명", "주소"]].drop_duplicates(),
    on=["대피소명", "주소"],
    how="inner",
)
earthquake_relations["재난유형"] = "지진"
earthquake_relations = earthquake_relations[["shelter_id", "재난유형"]]

tsunami_relations = shelters_table[["shelter_id", "대피소명", "주소"]].merge(
    tsunami_df[["대피소명", "주소"]].drop_duplicates(),
    on=["대피소명", "주소"],
    how="inner",
)
tsunami_relations["재난유형"] = "지진해일"
tsunami_relations = tsunami_relations[["shelter_id", "재난유형"]]

relation_source = pd.concat(
    [type_relations, earthquake_relations, tsunami_relations],
    ignore_index=True,
).drop_duplicates()

shelter_disaster_relations = relation_source.merge(
    disaster_types,
    on="재난유형",
    how="left",
)

shelter_disaster_relations = shelter_disaster_relations[
    ["shelter_id", "disaster_type_id"]
].drop_duplicates().reset_index(drop=True)

missing_region_in_shelters = int(shelters_table["region_id"].isna().sum())
missing_region_in_alerts = int(danger_alerts["region_id"].isna().sum())
missing_disaster_in_alerts = int(danger_alerts["disaster_type_id"].isna().sum())

if missing_region_in_shelters > 0 or missing_region_in_alerts > 0 or missing_disaster_in_alerts > 0:
    raise ValueError("There are missing foreign key values. Check the merge results before saving.")

db_file = get_project_path("preprocessing_code/data/simple_shelter_dashboard.db")
try:
    if os.path.exists(db_file):
        os.remove(db_file)
except PermissionError:
    db_file = get_project_path("preprocessing_code/data/simple_shelter_dashboard_copy.db")
    if os.path.exists(db_file):
        os.remove(db_file)

engine = create_engine("sqlite:///" + db_file.replace(os.sep, "/"))
Base.metadata.create_all(engine)

region_objects = [
    Region(
        region_id=int(row["region_id"]),
        sido=str(row["시도"]),
        sigungu=str(row["시군구"]),
    )
    for _, row in regions.iterrows()
]

disaster_type_objects = [
    DisasterType(
        disaster_type_id=int(row["disaster_type_id"]),
        disaster_name=str(row["재난유형"]),
    )
    for _, row in disaster_types.iterrows()
]

danger_alert_objects = [
    DangerAlert(
        alert_id=int(row["alert_id"]),
        announced_at=str(row["발표시간"]),
        region_id=int(row["region_id"]),
        disaster_type_id=int(row["disaster_type_id"]),
        alert_level=none_if_na(row["특보등급"]),
        affected_area=none_if_na(row["해당지역"]),
    )
    for _, row in danger_alerts.iterrows()
]

shelter_objects = [
    Shelter(
        shelter_id=int(row["shelter_id"]),
        region_id=int(row["region_id"]),
        shelter_name=str(row["대피소명"]),
        address=str(row["주소"]),
        shelter_type=none_if_na(row["대피소유형"]),
        latitude=float_if_na(row["위도"]),
        longitude=float_if_na(row["경도"]),
        earthquake_note=none_if_na(row["지진설명"]),
        capacity=int_if_na(row["수용인원"]),
    )
    for _, row in shelters_table.iterrows()
]

relation_objects = [
    ShelterDisasterRelation(
        shelter_id=int(row["shelter_id"]),
        disaster_type_id=int(row["disaster_type_id"]),
    )
    for _, row in shelter_disaster_relations.iterrows()
]

with Session(engine) as session:
    session.add_all(region_objects)
    session.add_all(disaster_type_objects)
    session.commit()

with Session(engine) as session:
    session.add_all(danger_alert_objects)
    session.add_all(shelter_objects)
    session.commit()

with Session(engine) as session:
    session.add_all(relation_objects)
    session.commit()
